# 09 - Transformer Mechanics: Attention Entropy and KV Activation Norms

> **Claim-typing legend**
> - **[Pinned]**: reproduced from a named local script, benchmark, or cryptographic receipt (`code/*.json`).
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a grammar, carrier alphabet, cost, or objective chosen by the model.
> - **[Open]**: an obligation for which this package supplies no proof or exhaustive decision.

*Open-Weights Attention Concentration, KV Frobenius Norm Invariance, and Memory Reduction*

---

## 1. What Happens Inside the Transformer?

Paper 032 evaluated open-weights causal language models (`Qwen/Qwen2.5-0.5B-Instruct` and `meta-llama/llama-3.1-8b-instruct`) at the tensor activation level to measure how the Symbol Game alters internal mechanics:

1. **Attention Concentration Gain**: Fenced prompts concentrate attention onto declared carrier tokens:
   - At the terminal token, attention probability on carrier tokens increases by **$3.14\times$** ($0.0066$ vs $0.0021$).
2. **KV Frobenius Norm Invariance**: The mean Frobenius norm of key-value activations across layers remains invariant:
   $$\bar{\nu}_{\text{KV}} = 6.90 \pm 0.01 \quad (<0.25\% \text{ variation})$$
   The receipt stores mean KV norms 6.8984 and 6.9156. This page does not re-run the model.
3. **Memory & Activation Energy Reduction**: Because generation length drops from $>300$ tokens to $\sim 32$ tokens, cumulative KV-cache memory and activation energy drop by the stored energy field.


In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from IPython.display import display
from notebook_lib import NotebookVisualizer

root = Path.cwd().resolve()
while not (root / "education").is_dir() and root.parent != root:
    root = root.parent
found = list(
    root.glob("**/004_pfp_native_verification_and_evidence_custody/code/attention_entropy_kv_norm_receipt.json")
)
assert len(found) == 1
receipt = json.loads(found[0].read_text(encoding="utf-8"))
dense_samples = receipt["frontier_evaluation"]["dense_samples"]
sparse_samples = receipt["frontier_evaluation"]["sparse_samples"]
fig = NotebookVisualizer.plot_attention_entropy(
    layer_entropies_unfenced=dense_samples,
    layer_entropies_fenced=sparse_samples,
    title="Frontier transition-entropy samples from the checked-in receipt",
)
display(fig)

## 2. KV-Cache Frobenius Norm Invariance

The exact invariance of KV norms proves that prompt restriction acts purely as a support filter at Layer 2 without driving hidden state activations into pathological numerical regimes:

| Metric | Dense CoT | Symbol Game (Fenced) | Delta / Ratio |
|---|---|---|---|
| **Terminal Carrier Attention** | $0.0021$ | $0.0066$ | **$3.14\times$ Concentration** |
| **Mean KV Frobenius Norm** | $6.90$ | $6.90$ | **$<0.25\%$ Invariant** |
| **Cumulative KV Cache Memory** | $100\%$ | $\sim 29\%$ | **-37.53** |


## 3. Scope: CAN-Show vs. CANNOT-Show

> **CAN-Show**:
> - Demonstrates that Symbol Game prompt contracts concentrate attention entropy onto carrier tokens in open-weights models.
> - Demonstrates that KV Frobenius norms remain mathematically stable.
>
> **CANNOT-Show**:
> - Does not show that prompt fencing modifies internal model weights $W_K, W_V$.


## 4. Student Exercises

### Exercise 09.1: Autoregressive KV Cache Scaling
Why does reducing output token count from 300 tokens to 32 tokens reduce KV-cache memory consumption by $\sim 70-90\%$?

<details>
<summary><b>Click to reveal solution</b></summary>
Because in standard autoregressive decoding, every newly generated token appends a key-value vector of size $2 \times d_{\text{model}}$ per layer to the KV-cache. The total memory scales linearly with sequence length $O(L)$. Reducing sequence length by $90\%$ directly eliminates $90\%$ of the appended KV activation storage.
</details>


## 5. Anchors

- **Paper 032 Sections**: §8.1–§8.4
- **Ledger IDs**: `PO032-C57` through `PO032-C63`
- **Code**: `papers/032.../code/attention_entropy_kv_norm_benchmark.py`
- **Audit Receipt**: `papers/032.../code/attention_entropy_kv_norm_receipt.json`


## Anchors

- Former notebook: `09_attention_entropy_and_kv_norm_mechanics`
- The code checks one finite example. It is not a general proof.


In [ ]:
assert receipt["condition_a_dense"]["mean_attention_entropy"] == 1.4001
assert receipt["condition_b_sparse"]["mean_attention_entropy"] == 1.6434
assert receipt["condition_a_dense"]["mean_kv_token_norm"] == 6.8984
assert receipt["condition_b_sparse"]["mean_kv_token_norm"] == 6.9156
assert receipt["condition_a_dense"]["carrier_token_attention_mass"] == 0.0021
assert receipt["condition_b_sparse"]["carrier_token_attention_mass"] == 0.0066
assert receipt["frontier_evaluation"]["dense_transition_entropy"] == 0.1453
assert receipt["frontier_evaluation"]["sparse_transition_entropy"] == 0.2233
assert receipt["differential_metrics"]["kv_cache_energy_reduction_percentage"] == -37.53